### Hugging Face


Based on: [Standford CS224N Tutorial Colab](https://colab.research.google.com/drive/1FyCMNTXfirWbJ18GuIT_JiTW0gwrxI3O?usp=sharing#scrollTo=RXm1K2sF-X88)

Setup

pip install transformers, datasets, accelerate

In [ ]:
from collections import defaultdict, Counter
import json

from matplotlib import pyplot as plt
import numpy as np

import torch

Helpers

In [ ]:
def print_encoding(model_inputs, indent=4):
    indent_str = " " * indent
    print("{")
    for k, v in model_inputs.items():
        print(indent_str + k + ":")
        print(indent_str + indent_str + str(v))
    print("}")

Intro

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification


tokenizer = AutoTokenizer.from_pretrained("siebert/sentiment-roberta-large-english")
model = AutoModelForSequenceClassification.from_pretrained("siebert/sentiment-roberta-large-english")


inputs = "I'm excited to learn about Hugging Face Transformers!"
tokens = tokenizer(inputs, return_tensor="pt") # pt, tf, np
"""
{
  'input_ids': tensor([[101, 1045, 2293, 9932, 102]]),
  'attention_mask': tensor([[1, 1, 1, 1, 1]])
}
"""
outputs = model(**tokens)


labels = ["NEGATIVE", "POSITIVE"]
prediction = torch.argmax(outputs.logits)


# print_encoding(tokenized_inputs)
# print(f"The prediction is {labels[prediction]}")

Tokenizers

In [ ]:
from transformers import DistilBertTokenizer, DistilBertTokenizerFast, AutoTokenizer


model_name = "distilbert/distilbert-base-cased"

Models


In [1]:
from transformers import AutoModelForSequenceClassification, DistilBertForSequenceClassification, DistilBertModel


base_model = DistilBertModel.from_pretrained("distilbert-base-cased") # only encoder, text->transformers->features

# model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-cased', num_labels=2)
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-cased', num_labels=2) # task-specific



from transformers import DistilBertConfig


config = DistilBertConfig()
config.num_labels = 2

model = DistilBertForSequenceClassification(config) # init with random weights

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 10098.97it/s]
DistilBertModel LOAD REPORT from: distilbert-base-cased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 11054.81it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | 

In [ ]:
input_str = " _ "
model_inputs = tokenizer(input_str, return_tensor="pt")

outputs = model(input_ids=model_inputs.input_ids, attention_mask=model_inputs.attention_mask)

model_outputs = model(**model_inputs)

In [ ]:
from torch.nn.functional import cross_entropy

label = torch.tensor([1])
loss = cross_entropy(model_outputs.logits, label)
loss.backward()

# list(model.named_parameters())[0]

Finetuning

In [ ]:
from datasets import load_dataset, DatasetDict


dataset_name = "stanfordnlp/imdb"
imdb_dataset = load_dataset(dataset_name)

def truncate(example):
    return {
        'text': " ".join(example['text'].split()[:50]),
        'label': example['label']
    }

# small_imdb_dataset = DatasetDict(
#     train=imdb_dataset['train'].shuffle(seed=1).select(range(128)).map(truncate),
#     val=imdb_dataset['train'].shuffle(seed=1).select(range(128, 160)).map(truncate),
# )
small_imdb_dataset = DatasetDict({
    "train": imdb_dataset['train'].shuffle(seed=1).select(range(128)).map(truncate),
    "val": imdb_dataset['train'].shuffle(seed=1).select(range(128, 160)).map(truncate),
})


# small_imdb_dataset['train'][:10]

In [ ]:
small_tokenized_dataset = small_imdb_dataset.map(
   lambda example: tokenizer(example['text'], padding=True, truncation=True),
   batched=True,
   batch_size=16
)
small_tokenized_dataset = small_tokenized_dataset.remove_columns(["text"])
small_tokenized_dataset = small_tokenized_dataset.rename_column("label", "labels")
small_tokenized_dataset.set_format("torch")


from torch.utils.data import DataLoader

train_dataloader = DataLoader(small_tokenized_dataset['train'], batch_size=16) # pyright: ignore[reportArgumentType]
eval_dataloader = DataLoader(small_tokenized_dataset['val'], batch_size=16) # pyright: ignore[reportArgumentType]

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-cased', num_labels=2)


from transformers import get_linear_schedule_with_warmup
from torch.optim import Adam
from tqdm.notebook import tqdm


num_epochs = 1
num_training_steps = len(train_dataloader)
optimizer = Adam(model.parameters(), lr=5e-5, weight_decay=0.01)
lr_scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)


best_val_loss = float("inf")
progress_bar = tqdm(range(num_training_steps))

for epoch in range(num_epochs):
    model.train()
    for batch_i, batch in enumerate(train_dataloader):
        output = model(**batch)
        optimizer.zero_grad()
        output.loss.backward()
        optimizer.step()
        lr_scheduler.step()
        progress_bar.update(1)
    
    model.eval()
    for batch_i, batch in enumerate(eval_dataloader):
        with torch.no_grad():
            output = model(**batch)
        loss += output.loss
    avg_val_loss = loss / len(eval_dataloader)

In [ ]:
from transformers import Trainer, TrainingArguments


arguments = TrainingArguments(
    output_dir="sample_hf_trainer",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    eval_strategy="epoch", # run validation at the end of each epoch
    save_strategy="epoch",
    learning_rate=2e-5,
    load_best_model_at_end=True,
    seed=224
)


def compute_metrics(eval_pred):
    """ """
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": np.mean(predictions == labels)}


trainer = Trainer(
    model=model,
    args=arguments,
    train_dataset=small_tokenized_dataset['train'],
    eval_dataset=small_tokenized_dataset['val'], # change to test when you do your final evaluation!
    compute_metrics=compute_metrics
)